# OSRT-605M midtrain3 — Colab GPU (drive from VSCode)

Continued pretraining of the v6 base toward Chinchilla-optimal. Resumes the
**same 12,600-step cosine** from wherever it left off — the latest checkpoint on
the private HF repo `HallD/osrt-v6-ckpt` (currently `step_2300`).

**How to run:** connect VSCode to a Colab runtime (Colab extension → *Connect to
Colab* → pick an **H100** or **A100** GPU runtime), then run these cells top to
bottom. The kernel lives on the Colab VM, so training runs on its GPU.

### Cross-session persistence (the important part)
The Colab VM disk is **ephemeral** and sessions have a 24h cap. `--hf-repo` makes
the run:
1. **pull** the latest `midtrain3_step_*` (+ the base) from HF on start, and
2. **push** every new checkpoint to HF as it saves (every 100 steps).

So a disconnect / reclaim loses at most ~100 steps — the next session's *Full run*
cell pulls the latest and continues the identical cosine. **Nothing lives only on
the VM.**

### Cost reality (read once)
This is a multi-billion-token pretraining grind: ~10,000 steps still to go to
reach 1× Chinchilla (~step 12,600). At Colab GPU rates that is **many sessions**.
Bank checkpoints, watch W&B, and **always run the Stop cell** when you step away —
idle GPU runtimes burn units. If a run misbehaves, stop it; the HF checkpoints are
safe regardless.

## 1 · GPU check
Confirm a CUDA GPU is attached before spending anything. If this shows *no GPU*,
reconnect the runtime to a GPU type and re-run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv || echo 'NO GPU — attach a GPU runtime'
import torch, sys
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')
assert torch.cuda.is_available(), 'Attach a GPU runtime before continuing.'

## 2 · Clone the repo + install pinned deps
The tokenizer (`v6_tokenizer_export/`) is committed, so a clone is all that's
needed. Versions are pinned to match the training code (torch 2.11 lineage).

In [ ]:
%cd /content
![ -d osrt ] || git clone https://github.com/CodeHalwell/OSRT-605M-A269M.git osrt
%cd /content/osrt
!git pull --ff-only 2>/dev/null; true
!pip install -q \
  transformers==5.3.0 datasets==4.6.1 tokenizers==0.22.2 safetensors==0.7.0 \
  wandb==0.25.1 lion-pytorch==0.2.4 huggingface_hub
print('repo + deps ready')

## 3 · Secrets
Paste your tokens. **Do not commit this notebook with tokens filled in** — clear
them before saving. `HF_TOKEN` needs write access to `HallD/osrt-v6-ckpt`.

> Tip: in Colab you can instead use the 🔑 *Secrets* panel and read them with
> `from google.colab import userdata; userdata.get('HF_TOKEN')` — safer than
> pasting into a cell.

In [ ]:
import os
os.environ['HF_TOKEN']        = ''  # <-- your HF write token
os.environ['WANDB_API_KEY']   = ''  # <-- your W&B key
os.environ['PYTHONPATH']      = 'src'
# Guards learned the hard way:
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '30'   # don't hang forever on a slow HF pull
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
assert os.environ['HF_TOKEN'] and os.environ['WANDB_API_KEY'], 'Fill in both tokens above.'
print('secrets set')

## 4 · Sanity gate — 30 steps
Always run this **first** on a fresh session. It exercises the full stack
(model build, data streams, optimizer, one checkpoint) in ~2-3 min without
touching the real run. If it errors, fix that before the full run — don't burn
GPU discovering a broken stream at step 500.

`--num-workers 0` is **mandatory on Colab**: the spawned streaming workers hit a
fatal `PyGILState_Release` teardown race that kills the run mid-stream.

In [ ]:
!python scripts/lightning_midtrain3.py --sanity \
  --ckpt-dir /content/ckpt \
  --tokenizer v6_tokenizer_export \
  --num-workers 0

## 5 · Full run — resumes from the latest HF checkpoint
This runs the real cosine. `--hf-repo` pulls the latest `midtrain3_step_*` (+ base)
from HF, the resume-scan continues the same 12,600-step schedule, and each new
checkpoint is pushed back to HF every 100 steps.

The cell **streams live output** and blocks while training — that's expected;
watch the `step N/12600 | task … | tok/s …` lines. If the VSCode connection drops,
the Colab kernel keeps running server-side; reconnect and the output reattaches.
If the *VM* is reclaimed, just re-run this cell next session — it resumes from HF.

Config baked in (`MidtrainExtend3Config`): seq 4096, eff-batch 66, peak LR 5e-5,
0.75 reasoning-dense mix, in-loop eval **off** (its dataset build stalls the GPU
and triggers idle reclaims), checkpoint every 100.

In [ ]:
!python scripts/lightning_midtrain3.py \
  --ckpt-dir /content/ckpt \
  --tokenizer v6_tokenizer_export \
  --hf-repo HallD/osrt-v6-ckpt \
  --ckpt-interval 100 \
  --num-workers 0

### Optional: run detached so it survives a VSCode disconnect cleanly
If you'd rather not hold the cell open, launch it in the background on the VM and
tail the log. Re-running the tail cell re-attaches to the live output.

In [ ]:
# launch detached (run once)
import subprocess, os
env = dict(os.environ)
subprocess.Popen(
    'python scripts/lightning_midtrain3.py --ckpt-dir /content/ckpt '
    '--tokenizer v6_tokenizer_export --hf-repo HallD/osrt-v6-ckpt '
    '--ckpt-interval 100 --num-workers 0 > /content/mt3.log 2>&1 &',
    shell=True, env=env)
print('launched in background → /content/mt3.log')

In [ ]:
# tail the background log (re-run anytime to see progress)
!tail -n 40 /content/mt3.log

## 6 · Monitor
- **W&B**: the `osrt-v6-midtrain3` run — watch `extend/task_loss` and `tok/s`.
- **HF**: `HallD/osrt-v6-ckpt` gains a new `osrt_v5_midtrain3_step_*.pt` every 100
  steps (the daemon prunes to the newest 3 remotely).
- Healthy signs: `drop=0`, `task` loss drifting down, `tok/s` steady (~7-9k on H100).

In [ ]:
# quick HF check — what's the latest banked checkpoint?
from huggingface_hub import HfApi
import re, os
fs = [f for f in HfApi().list_repo_files('HallD/osrt-v6-ckpt', repo_type='model',
                                          token=os.environ['HF_TOKEN'])
      if 'midtrain3_step' in f]
fs.sort(key=lambda f: int(re.search(r'(\d+)', f).group(1)))
print('latest on HF:', fs[-1] if fs else '(none)')

## 7 · STOP — always run when you step away
Idle GPU runtimes keep billing. Disconnect/delete the runtime from the VSCode
Colab panel (or Colab → *Runtime → Disconnect and delete runtime*). Checkpoints
are safe on HF, so stopping never loses progress.

In [ ]:
# kill any background training process on the VM (does not delete the runtime)
!pkill -f lightning_midtrain3 && echo 'training stopped' || echo 'nothing running'